# Constraint & Question Type Adaptation Analysis

Investigates which types of puzzles are easier or harder to adapt successfully.

**Datasets combined:**
- `results_combined_audited.csv` — adaptation results with judge verdicts + audit
- `Puzzle_Samples.xlsx` — original puzzles with `Solved_Percentile`
- Constraint annotation file — constraint types per puzzle
- Question type annotation file — question type per question

**Key questions:**
1. Which constraint types have the highest/lowest adaptation success rate?
2. Which question types are hardest to adapt?
3. Does puzzle difficulty (Solved_Percentile) correlate with adaptation success?
4. Are certain constraint × question type combinations systematically harder?
5. Which generators struggle most with specific constraint types?

## 1. Imports & Config

In [ ]:
import os
import json
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy.stats import pearsonr, kruskal
import warnings
warnings.filterwarnings('ignore')

# ── Paths — update these ──────────────────────────────────────────────────────
RESULTS_COMBINED    = '/content/results_combined_audited.csv'
ORIGINAL_PUZZLES    = '/content/puzzle_samples.xlsx'
CONSTRAINT_ANN      = '/content/annotated_puzzles_wide_final.csv'   # has Puzzle_Number, constraint_annotation, constraint_types
QUESTION_TYPE_ANN   = '/content/annotated_question_types_final.csv' # has Puzzle_Number, question_col, question_number, question_type

OUTDIR = './outputs/constraint_analysis'
os.makedirs(OUTDIR, exist_ok=True)

PASS_OUTCOMES = ['equivalent', 'rescued_formatting']

## 2. Load & Merge All Datasets

In [ ]:

# ── Adaptation results ────────────────────────────────────────────────────────
results = pd.read_csv(RESULTS_COMBINED)
results = results[results['generator_model'] != 'deepseek-r1-distill-qwen-7b']
results = results.drop(columns=['Solved_Percentile', 'difficulty'], errors='ignore')
results['passed'] = results['corrected_outcome'].isin(PASS_OUTCOMES)
print(f'Adaptation rows     : {len(results)}')
print(f'Unique puzzles      : {results[["Puzzle_Number","Question_Number"]].drop_duplicates().shape[0]}')
print(f'Generators          : {results["generator_model"].nunique()}')

# ── Original puzzles ──────────────────────────────────────────────────────────
orig = pd.read_excel(ORIGINAL_PUZZLES)[[
    'Puzzle_Number', 'Question_Number', 'Sample', 'Solved_Percentile'
]].drop_duplicates(subset=['Puzzle_Number', 'Question_Number'])

def map_difficulty(p):
    if p < 30:   return 'Hard'
    elif p < 60: return 'Medium'
    else:        return 'Easy'

orig['difficulty'] = orig['Solved_Percentile'].apply(map_difficulty)

# ── Constraint annotations ────────────────────────────────────────────────────
const = pd.read_csv(CONSTRAINT_ANN)

def parse_constraint_types(val):
    if pd.isna(val):
        return []
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except Exception:
        return [str(val)]

const['constraint_types_list'] = const['constraint_types'].apply(parse_constraint_types)
const['dominant_constraint']   = const['constraint_types_list'].apply(
    lambda x: x[0] if len(x) > 0 else 'unknown'
)
const['n_constraint_types'] = const['constraint_types_list'].apply(len)

# ── Question type annotations ─────────────────────────────────────────────────
qtype = pd.read_csv(QUESTION_TYPE_ANN)

if 'question_number' in qtype.columns and 'Question_Number' not in qtype.columns:
    qtype = qtype.rename(columns={'question_number': 'Question_Number'})
if 'question_col' in qtype.columns and 'Question_Number' not in qtype.columns:
    qtype['Question_Number'] = qtype['question_col']

# Normalize to Q1/Q2/Q3 format
qtype['Question_Number'] = qtype['Question_Number'].astype(str).apply(
    lambda x: x if x.startswith('Q') else f'Q{x}'
)

# ── Merge all ─────────────────────────────────────────────────────────────────
df = results.merge(
    orig[['Puzzle_Number', 'Question_Number', 'Solved_Percentile', 'difficulty']],
    on=['Puzzle_Number', 'Question_Number'], how='left'
).merge(
    const[['Puzzle_Number', 'constraint_types_list', 'dominant_constraint', 'n_constraint_types']],
    on='Puzzle_Number', how='left'
).merge(
    qtype[['Puzzle_Number', 'Question_Number', 'question_type']],
    on=['Puzzle_Number', 'Question_Number'], how='left'
)

print(f'\nMerged rows         : {len(df)}')
print(f'Missing difficulty  : {df["Solved_Percentile"].isna().sum()}')
print(f'Missing constraint  : {df["dominant_constraint"].isna().sum()}')
print(f'Missing q_type      : {df["question_type"].isna().sum()}')
print(f'\nOverall pass rate   : {df["passed"].mean():.1%}')

In [ ]:
print('df Question_Number sample:', df['Question_Number'].head().tolist())
print('qtype Question_Number sample:', qtype['Question_Number'].astype(str).head().tolist())

## 3. Per-Puzzle Adaptation Success Rate

In [ ]:
# Aggregate to puzzle level — success rate across all generators
puzzle_agg = df.groupby(['Puzzle_Number', 'Question_Number']).agg(
    n_attempts          = ('passed', 'count'),
    n_passed            = ('passed', 'sum'),
    adaptation_success  = ('passed', 'mean'),
    difficulty          = ('difficulty', 'first'),
    Solved_Percentile   = ('Solved_Percentile', 'first'),
    dominant_constraint = ('dominant_constraint', 'first'),
    n_constraint_types  = ('n_constraint_types', 'first'),
    question_type       = ('question_type', 'first'),
).reset_index()

puzzle_agg['adaptation_success_%'] = puzzle_agg['adaptation_success'] * 100

print(f'Unique puzzles: {len(puzzle_agg)}')
print(f'\nAdaptation success rate distribution:')
print(puzzle_agg['adaptation_success_%'].describe().round(1).to_string())
print(f'\nFully passed (100%)  : {(puzzle_agg["adaptation_success"] == 1.0).sum()}')
print(f'Partially passed     : {((puzzle_agg["adaptation_success"] > 0) & (puzzle_agg["adaptation_success"] < 1)).sum()}')
print(f'Fully failed (0%)    : {(puzzle_agg["adaptation_success"] == 0.0).sum()}')

puzzle_agg.to_csv(os.path.join(OUTDIR, 'puzzle_adaptation_rates.csv'), index=False)

## 4. Constraint Type Analysis

In [ ]:
# Expand constraint_types_list — each row gets one constraint type
# for a puzzle with [ordering, adjacency], we get two rows
expanded = df.explode('constraint_types_list').rename(
    columns={'constraint_types_list': 'constraint_type'}
)
expanded = expanded[expanded['constraint_type'].notna()]

# Success rate per constraint type
const_agg = expanded.groupby('constraint_type').agg(
    n_puzzle_attempts = ('passed', 'count'),
    pass_rate         = ('passed', 'mean'),
).reset_index()
const_agg['pass_rate_%'] = const_agg['pass_rate'] * 100
const_agg = const_agg.sort_values('pass_rate_%', ascending=False)

print('Adaptation success rate by constraint type:')
print(const_agg.to_string(index=False))
const_agg.to_csv(os.path.join(OUTDIR, 'constraint_type_pass_rates.csv'), index=False)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
norm   = plt.Normalize(vmin=const_agg['pass_rate_%'].min(), vmax=const_agg['pass_rate_%'].max())
colors = [plt.cm.Blues(norm(v) * 0.6 + 0.3) for v in const_agg['pass_rate_%']]
bars   = ax.barh(const_agg['constraint_type'], const_agg['pass_rate_%'],
                 color=colors, edgecolor='#444', linewidth=0.6)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
for bar, (_, row) in zip(bars, const_agg.iterrows()):
    ax.text(1, bar.get_y() + bar.get_height()/2,
            f'n={row["n_puzzle_attempts"]}', va='center', fontsize=7.5, color='#555')
ax.set_xlabel('Adaptation Pass Rate (%)', fontweight='bold')
ax.set_title('Adaptation Success Rate by Constraint Type\n(across all generators)',
             fontweight='bold')
ax.set_xlim(0, 115)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'constraint_type_pass_rates.png'), dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

## 5. Question Type Analysis

In [ ]:
qtype_agg = df.groupby('question_type').agg(
    n_attempts = ('passed', 'count'),
    pass_rate  = ('passed', 'mean'),
).reset_index()
qtype_agg['pass_rate_%'] = qtype_agg['pass_rate'] * 100
qtype_agg = qtype_agg.sort_values('pass_rate_%', ascending=False)

print('Adaptation success rate by question type:')
print(qtype_agg.to_string(index=False))
qtype_agg.to_csv(os.path.join(OUTDIR, 'question_type_pass_rates.csv'), index=False)

fig, ax = plt.subplots(figsize=(10, 5))
norm   = plt.Normalize(vmin=qtype_agg['pass_rate_%'].min(), vmax=qtype_agg['pass_rate_%'].max())
colors = [plt.cm.Blues(norm(v) * 0.6 + 0.3) for v in qtype_agg['pass_rate_%']]
bars   = ax.barh(qtype_agg['question_type'], qtype_agg['pass_rate_%'],
                 color=colors, edgecolor='#444', linewidth=0.6)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
for bar, (_, row) in zip(bars, qtype_agg.iterrows()):
    ax.text(1, bar.get_y() + bar.get_height()/2,
            f'n={row["n_attempts"]}', va='center', fontsize=7.5, color='#555')
ax.set_xlabel('Adaptation Pass Rate (%)', fontweight='bold')
ax.set_title('Adaptation Success Rate by Question Type\n(across all generators)',
             fontweight='bold')
ax.set_xlim(0, 115)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'question_type_pass_rates.png'), dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

## 6. Difficulty vs Adaptation Success

In [ ]:
# Scatter: Solved_Percentile vs adaptation_success_rate
valid = puzzle_agg[puzzle_agg['Solved_Percentile'].notna()]
r, p  = pearsonr(valid['Solved_Percentile'], valid['adaptation_success'])
p_str = 'p<0.001' if p < 0.001 else f'p={p:.3f}'

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
ax = axes[0]
colors_diff = {'Hard': '#d62728', 'Medium': '#ff7f0e', 'Easy': '#2ca02c'}
for diff, g in valid.groupby('difficulty'):
    ax.scatter(g['Solved_Percentile'], g['adaptation_success_%'],
               c=colors_diff[diff], label=diff, alpha=0.6, s=60, edgecolors='white', linewidth=0.4)

# Trend line
z = np.polyfit(valid['Solved_Percentile'], valid['adaptation_success_%'], 1)
xline = np.linspace(valid['Solved_Percentile'].min(), valid['Solved_Percentile'].max(), 100)
ax.plot(xline, np.poly1d(z)(xline), 'navy', linewidth=1.5, linestyle='--', alpha=0.7)

ax.set_xlabel('Human Solved Percentile (%)', fontweight='bold')
ax.set_ylabel('Adaptation Success Rate (%)', fontweight='bold')
ax.set_title(f'Human Difficulty vs Adaptation Success\n(r={r:.3f}, {p_str})', fontweight='bold')
ax.legend(title='Difficulty')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.25)

# Box plot by difficulty tier
ax = axes[1]
DIFF_ORDER = ['Hard', 'Medium', 'Easy']
data_by_diff = [valid[valid['difficulty'] == d]['adaptation_success_%'].values for d in DIFF_ORDER]
bp = ax.boxplot(data_by_diff, labels=DIFF_ORDER, patch_artist=True,
                medianprops={'color': 'black', 'linewidth': 2})
for patch, diff in zip(bp['boxes'], DIFF_ORDER):
    patch.set_facecolor(colors_diff[diff])
    patch.set_alpha(0.6)

# Overlay means
for i, d in enumerate(DIFF_ORDER):
    mean_val = valid[valid['difficulty'] == d]['adaptation_success_%'].mean()
    ax.scatter(i + 1, mean_val, color='black', s=60, zorder=5, marker='D')
    ax.text(i + 1, mean_val + 1.5, f'{mean_val:.1f}%', ha='center', fontsize=9)

ax.set_ylabel('Adaptation Success Rate (%)', fontweight='bold')
ax.set_title('Adaptation Success by Difficulty Tier', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'difficulty_vs_adaptation.png'), dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

print('Mean adaptation success by difficulty:')
print(valid.groupby('difficulty')['adaptation_success_%'].mean().round(1).to_string())

## 7. Constraint Type × Question Type Heatmap

In [ ]:
# Cross-tab: dominant constraint × question type
cross = df[df['dominant_constraint'].notna() & df['question_type'].notna()]
pivot = cross.groupby(['dominant_constraint', 'question_type'])['passed'].agg(
    ['mean', 'count']
).reset_index()
pivot['pass_rate_%'] = pivot['mean'] * 100

# Only include combos with at least 3 samples
pivot = pivot[pivot['count'] >= 3]

heatmap_data = pivot.pivot(index='dominant_constraint', columns='question_type', values='pass_rate_%')
count_data   = pivot.pivot(index='dominant_constraint', columns='question_type', values='count')

fig, ax = plt.subplots(figsize=(max(10, len(heatmap_data.columns) * 1.5),
                                max(6,  len(heatmap_data.index)  * 0.7)))

sns.heatmap(
    heatmap_data,
    annot=False,
    cmap='Blues', vmin=0, vmax=100,
    linewidths=0.5, linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Pass Rate (%)'}
)

# Annotate with pass% and n
for i, row_idx in enumerate(heatmap_data.index):
    for j, col_idx in enumerate(heatmap_data.columns):
        val = heatmap_data.loc[row_idx, col_idx]
        cnt = count_data.loc[row_idx, col_idx] if row_idx in count_data.index and col_idx in count_data.columns else np.nan
        if not np.isnan(val):
            text_color = 'white' if val > 60 else 'black'
            ax.text(j + 0.5, i + 0.4, f'{val:.0f}%',
                    ha='center', va='center', fontsize=8.5,
                    color=text_color, fontweight='bold')
            if not np.isnan(cnt):
                ax.text(j + 0.5, i + 0.65, f'n={int(cnt)}',
                        ha='center', va='center', fontsize=6.5, color=text_color)

ax.set_xlabel('Question Type', fontweight='bold', fontsize=11)
ax.set_ylabel('Dominant Constraint Type', fontweight='bold', fontsize=11)
ax.set_title('Adaptation Pass Rate: Constraint Type × Question Type\n(cells with n<3 omitted)',
             fontweight='bold', fontsize=12)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'constraint_x_question_heatmap.png'), dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

## 8. Generator × Constraint Type Heatmap

In [ ]:
# Which generators struggle with which constraint types?
gen_const = expanded.groupby(['generator_model', 'constraint_type'])['passed'].agg(
    ['mean', 'count']
).reset_index()
gen_const['pass_rate_%'] = gen_const['mean'] * 100
gen_const = gen_const[gen_const['count'] >= 3]

gen_const_pivot = gen_const.pivot(index='generator_model', columns='constraint_type', values='pass_rate_%')
gen_const_count = gen_const.pivot(index='generator_model', columns='constraint_type', values='count')

fig, ax = plt.subplots(figsize=(max(12, len(gen_const_pivot.columns) * 1.2),
                                max(6,  len(gen_const_pivot.index)  * 0.8)))

sns.heatmap(
    gen_const_pivot,
    annot=False,
    cmap='Blues', vmin=0, vmax=100,
    linewidths=0.5, linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Pass Rate (%)'}
)

for i, row_idx in enumerate(gen_const_pivot.index):
    for j, col_idx in enumerate(gen_const_pivot.columns):
        val = gen_const_pivot.loc[row_idx, col_idx]
        cnt = gen_const_count.loc[row_idx, col_idx] if row_idx in gen_const_count.index and col_idx in gen_const_count.columns else np.nan
        if not np.isnan(val):
            text_color = 'white' if val > 60 else 'black'
            ax.text(j + 0.5, i + 0.4, f'{val:.0f}%',
                    ha='center', va='center', fontsize=8,
                    color=text_color, fontweight='bold')
            if not np.isnan(cnt):
                ax.text(j + 0.5, i + 0.68, f'n={int(cnt)}',
                        ha='center', va='center', fontsize=6, color=text_color)

ax.set_xlabel('Constraint Type', fontweight='bold', fontsize=11)
ax.set_ylabel('Generator Model', fontweight='bold', fontsize=11)
ax.set_title('Adaptation Pass Rate: Generator × Constraint Type',
             fontweight='bold', fontsize=12)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'generator_x_constraint_heatmap.png'), dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

## 9. Complexity Analysis — Number of Constraint Types

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# -------------------------
# Data prep
# -------------------------
complexity = df[df['n_constraint_types'].notna()].copy()
complexity['n_constraint_types'] = complexity['n_constraint_types'].astype(int)

comp_agg = complexity.groupby('n_constraint_types').agg(
    n_attempts=('passed', 'count'),
    pass_rate=('passed', 'mean'),
).reset_index()

comp_agg['pass_rate_%'] = comp_agg['pass_rate'] * 100

# Correlation
r, p = pearsonr(
    complexity['n_constraint_types'],
    complexity['passed'].astype(int)
)

p_str = 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'

print('Pass rate by number of constraint types:')
print(comp_agg.to_string(index=False))
print(f'\nPearson r = {r:.3f} ({p_str})')

# -------------------------
# Plot
# -------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))

# Soft academic blue palette
colors = plt.cm.Blues(np.linspace(0.45, 0.75, len(comp_agg)))

bars = ax.bar(
    comp_agg['n_constraint_types'],
    comp_agg['pass_rate_%'],
    color=colors,
    edgecolor='black',
    linewidth=0.8,
    width=0.6
)

# ONLY percentage labels (clean)
ax.bar_label(
    bars,
    fmt='%.1f%%',
    padding=3,
    fontsize=10,
    color='black'
)

# X-axis labels include sample size (no clutter on bars)
ax.set_xticks(comp_agg['n_constraint_types'])
ax.set_xticklabels([
    f'{row["n_constraint_types"]}\n(n={row["n_attempts"]})'
    for _, row in comp_agg.iterrows()
], fontsize=10)

# Labels
ax.set_xlabel('Number of Distinct Constraint Types', fontsize=11, fontweight='bold')
ax.set_ylabel('Adaptation Success Rate (%)', fontsize=11, fontweight='bold')

ax.set_title(
    f'Puzzle Complexity vs. Adaptation Success\n'
    f'(r = {r:.3f}, {p_str})',
    fontsize=12,
    fontweight='bold'
)

# Styling (ACL-like)
ax.set_ylim(0, 105)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.25)

ax.tick_params(axis='both', labelsize=10)

plt.tight_layout()

# Save high-res figure
plt.savefig(
    os.path.join(OUTDIR, 'complexity_vs_adaptation.png'),
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)

plt.show()

## 10. Failure Analysis — What Goes Wrong?

In [ ]:
# For failed adaptations, look at failure_nature + r1_feedback
failed = df[~df['passed']].copy()

print(f'Failed adaptation rows: {len(failed)}')
print(f'\nFailure nature breakdown:')
print(failed['failure_nature'].value_counts().to_string())

print(f'\nFailure nature by constraint type (dominant):')
fn_cross = failed.groupby(['dominant_constraint', 'failure_nature']).size().unstack(fill_value=0)
print(fn_cross.to_string())

print(f'\nFailure nature by question type:')
fn_qtype = failed.groupby(['question_type', 'failure_nature']).size().unstack(fill_value=0)
print(fn_qtype.to_string())

fn_cross.to_csv(os.path.join(OUTDIR, 'failure_by_constraint.csv'))
fn_qtype.to_csv(os.path.join(OUTDIR, 'failure_by_question_type.csv'))

## 11. Statistical Tests — Is Constraint Type Effect Significant?

In [ ]:
# Kruskal-Wallis test across constraint types
groups_const = [
    expanded[expanded['constraint_type'] == ct]['passed'].astype(int).values
    for ct in expanded['constraint_type'].dropna().unique()
    if expanded[expanded['constraint_type'] == ct].shape[0] >= 5
]

if len(groups_const) >= 2:
    stat, p = kruskal(*groups_const)
    print(f'Kruskal-Wallis test across constraint types:')
    print(f'  H={stat:.3f}, p={p:.4f}')
    print(f'  {"Significant" if p < 0.05 else "Not significant"} at α=0.05')

# Kruskal-Wallis across question types
groups_qtype = [
    df[df['question_type'] == qt]['passed'].astype(int).values
    for qt in df['question_type'].dropna().unique()
    if df[df['question_type'] == qt].shape[0] >= 5
]

if len(groups_qtype) >= 2:
    stat, p = kruskal(*groups_qtype)
    print(f'\nKruskal-Wallis test across question types:')
    print(f'  H={stat:.3f}, p={p:.4f}')
    print(f'  {"Significant" if p < 0.05 else "Not significant"} at α=0.05')

## 12. Summary Table — All Dimensions Combined

In [ ]:
print('='*65)
print('ADAPTATION SUCCESS SUMMARY')
print('='*65)

print('\n── By Constraint Type ──────────────────────────────────────')
print(const_agg[['constraint_type', 'pass_rate_%', 'n_puzzle_attempts']].to_string(index=False))

print('\n── By Question Type ────────────────────────────────────────')
print(qtype_agg[['question_type', 'pass_rate_%', 'n_attempts']].to_string(index=False))

print('\n── By Difficulty ───────────────────────────────────────────')
diff_summary = df.groupby('difficulty')['passed'].agg(['mean','count'])
diff_summary['pass_rate_%'] = diff_summary['mean'] * 100
print(diff_summary[['pass_rate_%','count']].round(1).to_string())

print('\n── By Number of Constraint Types ───────────────────────────')
print(comp_agg[['n_constraint_types','pass_rate_%','n_attempts']].to_string(index=False))

print('\n── Hardest Puzzles to Adapt (bottom 10 by success rate) ────')
hard = puzzle_agg.sort_values('adaptation_success_%').head(10)
print(hard[['Puzzle_Number','Question_Number','difficulty','dominant_constraint',
            'question_type','n_attempts','adaptation_success_%']].to_string(index=False))

print('\n── Easiest Puzzles to Adapt (top 10 by success rate) ───────')
easy = puzzle_agg.sort_values('adaptation_success_%', ascending=False).head(10)
print(easy[['Puzzle_Number','Question_Number','difficulty','dominant_constraint',
            'question_type','n_attempts','adaptation_success_%']].to_string(index=False))

In [ ]:
from google.colab import files
import shutil

# Zip the folder
shutil.make_archive('my_outputs', 'zip', '/content/outputs')

# Download
files.download('my_outputs.zip')